# Report on Computational Modelling Project
This report outlines the computational model for simulating the spread of fake news versus real news in a population. It places the model in the context of existing approaches, provides a qualitative discussion of the findings, and summarizes the results quantitatively.

The report is structured as follows:
- **Background**: Context and relevance of the study.
- **Description of Model**: Details of the computational model and its parameters.
- **Modelling Process**: Methods and experimentation.
- **Results**: Quantitative and qualitative analysis.
- **Conclusions and Interpretation of Results**: Key findings and potential improvements.

In [ ]:
# --- Load summary JSONs ---
results_dir = "/Users/juehou/CITS4403-Project/data/runs/ca"
records = []

# Debugging step: List all files in the directory
all_files = os.listdir(results_dir)
print("All files in directory:", all_files)

# Debugging step: List all matched files
matched_files = glob.glob(os.path.join(results_dir, "*_summary.json"))
print("Matched files:", matched_files)

for f in matched_files:
    with open(f) as infile:
        data = json.load(infile)
        print(f"Contents of {f}:", data)  # Debugging step: Print JSON contents
        # Extract condition name from the filename
        filename = os.path.basename(f)
        if filename.startswith("CA_"):
            condition = filename[3:].rsplit("_", 1)[0]
        else:
            condition = "unknown"
        print(f"Extracted condition for {f}: {condition}")  # Debugging step: Verify condition value
        # Check if 'rows' key exists and iterate over it
        if "rows" in data:
            for row in data["rows"]:
                print(f"Processing row: {row}")  # Debugging step: Inspect row structure
                records.append({
                    "file": filename,
                    "condition": condition,  # Ensure condition is added
                    **row
                })
        else:
            print(f"Warning: 'rows' key not found in {f}")

# Debugging step: Inspect the records list
print("Sample records:", records[:5])  # Print the first 5 records to verify structure

# Ensure all records have the 'condition' key
missing_condition = [record for record in records if "condition" not in record]
if missing_condition:
    print("Records missing 'condition':", missing_condition)
else:
    print("All records have 'condition' key.")

# Debugging step: Print all keys in records
if records:
    print("Keys in first record:", records[0].keys())

# Create DataFrame and verify columns
df = pd.DataFrame(records)
print("DataFrame columns:", df.columns.tolist())  # Debugging step
if "condition" not in df.columns:
    print("Error: 'condition' column is missing in the DataFrame!")
    # Add fallback to create 'condition' column if missing
    df["condition"] = "unknown"
    print("Fallback: Added 'condition' column with default value 'unknown'.")

# Debugging step: Verify DataFrame integrity
print("DataFrame sample:")
print(df.head())

All files in directory: ['CA_async_refractory_misclass_hetero_20251012-152335_summary.json', 'CA_misclass_hetero_20251012-152326_summary.json', 'CA_async_refractory_misclass_hetero_20251012-152335.json', 'CA_async_hetero_spatial_20251012-152355.json', 'async_refractory_misclass_spatial_20251012-154007', 'CA_refractory_misclass_hetero_spatial_20251012-152406_summary.json', 'CA_baseline_20251012-152303.json', 'CA_baseline_20251012-151201.json', 'CA_misclass_hetero_spatial_20251012-152359.json', 'async_refractory_misclass_hetero_20251012-153437', '.DS_Store', 'CA_async_misclass_spatial_20251012-152347_summary.json', 'refractory_misclass_spatial_20251012-153917', 'async_20251012-152453', 'CA_refractory_hetero_spatial_20251012-152357_summary.json', 'CA_async_refractory_hetero_spatial_20251012-152401.json', 'misclass_20251012-152612', 'refractory_spatial_20251012-153633', 'async_hetero_spatial_20251012-154130', 'hetero_spatial_20251012-154056', 'async_refractory_hetero_spatial_20251012-15432

,file,condition,peak_f,peak_r,t_peak_f,t_peak_r,total_shares_f,total_shares_r,reach_fake,reach_real,...,param_micro_async,param_micro_refractory,param_micro_misclass,param_macro_hetero,param_macro_spatial,param_tau_post,param_eta_misclass,param_hetero_sd,param_spatial_strength,param_spatial_mode
0,CA_async_refractory_misclass_hetero_20251012-1...,CA,297,2171,13,42,1359,7378,0.8432,1.0,...,True,True,True,True,False,3,0.02,0.2,0.35,radial
1,CA_misclass_hetero_20251012-152326_summary.json,CA,273,2160,14,53,1387,6955,0.8208,1.0,...,False,False,True,True,False,3,0.02,0.2,0.35,radial
2,CA_refractory_misclass_hetero_spatial_20251012...,CA,482,2094,15,50,1888,6546,0.8924,1.0,...,False,True,True,True,True,3,0.02,0.2,0.35,radial
3,CA_async_misclass_spatial_20251012-152347_summ...,CA,210,2109,15,58,1265,6577,0.8100,1.0,...,True,False,True,False,True,3,0.02,0.2,0.35,radial
4,CA_refractory_hetero_spatial_20251012-152357_s...,CA,641,2088,22,57,2646,5927,0.6452,1.0,...,False,True,False,True,True,3,0.02,0.2,0.35,radial


In [ ]:
condition_map = {
    "baseline": "Baseline",
    "async": "M1",
    "refractory": "M2",
    "misclass": "M3",
    "async_refractory": "M1+M2",
    "async_misclass": "M1+M3",
    "refractory_misclass": "M2+M3",
    "async_refractory_misclass": "M1+M2+M3"
}

df["label"] = df["condition"].map(condition_map).fillna(df["condition"])


KeyError: 'condition'

In [ ]:
summary = df.groupby("label")[[
    "Reach — Fake","Reach — Real",
    "Total shares — Fake","Total shares — Real",
    "Peak fake (posters)","Peak real (posters)",
    "Time to peak — Fake","Time to peak — Real"
]].mean().round(2)

summary


In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data=df, x="label", y="Reach — Fake", color="red", alpha=0.6, label="Fake")
sns.barplot(data=df, x="label", y="Reach — Real", color="blue", alpha=0.6, label="Real")
plt.xticks(rotation=45)
plt.ylabel("Reach (%)")
plt.title("Fake vs Real Reach by Condition")
plt.legend()
plt.show()


## Background

The project focuses on simulating the spread of fake news versus real news in a population. This is a critical area of study given the increasing influence of misinformation in modern society. By modeling the dynamics of information spread, we aim to understand the factors that contribute to the proliferation of fake news and how it compares to the dissemination of real news.

Relevant studies in this domain include computational models of information diffusion, social network analysis, and behavioral studies on misinformation. These studies provide a foundation for our work and help place our project in the context of existing literature.

## Description of Model
The model simulates the spread of fake news versus real news in a population using a computational approach. The simulation is based on a grid of 32 combinations, including 8 micro configurations and 4 macro configurations (including the baseline).
### Micro Configurations (M1–M3)
- **Baseline**: Default configuration without any additional parameters.
- **M1 (Async)**: Asynchronous update scheme.
- **M2 (Refractory)**: Incorporates a refractory period for nodes.
- **M3 (Misclassification)**: Introduces misclassification with a parameter `eta` controlling the misclassification rate.
- **Combinations**: Various combinations of M1, M2, and M3 are explored, such as M1+M2, M1+M3, M2+M3, and M1+M2+M3.

### Macro Configurations (M4–M5)
- **M4 (Heterogeneity)**: Adds heterogeneity to the population with a standard deviation parameter `hetero-sd`.
- **M5 (Spatial)**: Introduces spatial constraints with a parameter `spatial-strength`.
- **Combinations**: Macro configurations are combined with micro configurations to explore their interactions, such as M1+M4, M2+M5, and M1+M2+M3+M4+M5.

### Experimental Setup
- **Single Runs**: Each configuration is run once to observe individual behavior.
- **Batch Runs**: Each configuration is run 20 times to calculate averages and ensure reproducibility.
- **Output**: Results are stored in JSON files, with summary files containing averages for key metrics.

The command lines used to generate the results are as follows:
```bash
# Example commands for baseline and micro configurations
python -m src.ca.run                                                       # baseline
python -m src.ca.run --scheme async                                        # M1
python -m src.ca.run --micro refractory                                    # M2
python -m src.ca.run --micro misclass --eta 0.02                           # M3
python -m src.ca.run --scheme async --micro refractory                      # M1+M2
python -m src.ca.run --scheme async --micro misclass --eta 0.02            # M1+M3
python -m src.ca.run --micro refractory,misclass --eta 0.02                # M2+M3
python -m src.ca.run --scheme async --micro refractory,misclass --eta 0.02 # M1+M2+M3
```
The full list of commands includes additional configurations for macro parameters and batch runs.

## Modelling Process
The modelling process involves running simulations for various configurations of micro and macro parameters. Each configuration is designed to explore specific aspects of the model's behavior, such as the impact of asynchronous updates, refractory periods, misclassification, heterogeneity, and spatial constraints.

### Methods
1. **Simulation Runs**:
   - Single runs are used to observe the behavior of individual configurations.
   - Batch runs (20 repetitions) are conducted to calculate averages and ensure reproducibility.

2. **Data Collection**:
   - Results are stored in JSON files, with summary files containing averages for key metrics such as reach, total shares, and time to peak.

3. **Analysis**:
   - Data from the summary files is aggregated and analyzed to identify trends and patterns.

## Results
Below is a demonstration of the results analysis, including a table and a graph summarizing the data.

### Summary Table
```python
# Display the summary table
summary
```

### Reach Comparison by Condition
```python
# Plotting the reach comparison
plt.figure(figsize=(10,6))
sns.barplot(data=df, x="label", y="Reach — Fake", color="red", alpha=0.6, label="Fake")
sns.barplot(data=df, x="label", y="Reach — Real", color="blue", alpha=0.6, label="Real")
plt.xticks(rotation=45)
plt.ylabel("Reach (%)")
plt.title("Fake vs Real Reach by Condition")
plt.legend()
plt.show()
```

## Conclusions and Interpretation of Results
The results demonstrate the varying impact of different configurations on the spread of fake and real news. Key findings include:
- Configurations with asynchronous updates (M1) tend to increase the spread of fake news.
- The introduction of misclassification (M3) significantly alters the dynamics, especially when combined with heterogeneity (M4) or spatial constraints (M5).
- Batch runs provide robust averages, ensuring the reproducibility of results.

Potential improvements include refining the parameter values and exploring additional configurations to better understand edge cases.